# World Happiness Report: Data Modeling Pipeline

This notebook builds a full modeling pipeline on the enriched World Happiness Report dataset — WHR data (2005–present) merged with socioeconomic factos from the Wordl Bank — to identify which factors most strongly predict national happiness scores (`Life Ladder`), both globally and regionally.

The pipeline runs in four phases:

1. **Statistical Pre-Screening** — skewness correction and multicollinearity pruning (VIF) to produce a clean, non-redundant feature set before any model sees the data.
2. **Champion Model Selection** — GridSearchCV over polynomial degree and regularisation strength to find the best-fitting linear model per scope.
3. **Permutation Importance** — model-agnostic feature ranking that measures how much each feature actually contributes to predictive accuracy, as opposed to what a correlation coefficient would suggest.
4. **Global → Regional Comparison** — the full pipeline runs once on all countries combined, then independently on each geographic region, so that drivers of happiness hidden at the global level can surface regionally.

WHR-native columns (e.g. *Social Support*, *Freedom to Make Life Choices*) are excluded throughout to avoid data leakage — the goal is to explain happiness from external socioeconomic and governance indicators alone.

## 0 · Initial setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})


In [ ]:
df = pd.read_csv('cleaned-datasets/HZ_WH_data.csv')

TARGET     = 'Life Ladder'
ID_COLS    = ['Country Name', 'ISO3', 'Regional Indicator', 'Year']
SKEW_THRESH = 0.5
VIF_THRESH  = 10.0

ALL_FEATURES = [c for c in df.select_dtypes(include=np.number).columns
                if c not in [TARGET] + ID_COLS]

print(f"Shape: {df.shape}")
print(f"Features ({len(ALL_FEATURES)}): {ALL_FEATURES}")

---
---

This scatterplot is used to test, by eye, whether each variable is linearly, logarithmically or not correlated with the happiness score (Life Ladder) 

In [ ]:
# raw scatter — before any transforms
ncols = 4
nrows = (len(ALL_FEATURES) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3))
axes = axes.flatten()

for i, feat in enumerate(ALL_FEATURES):
    axes[i].scatter(df[feat], df[TARGET], alpha=0.4, edgecolors='none')
    axes[i].set_xlabel(feat, fontsize=8)
    axes[i].set_ylabel(TARGET, fontsize=8)
    axes[i].set_title(feat, fontsize=9, fontweight='bold')
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f'Feature vs {TARGET} — Raw (pre-transform)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/HZ_feature_vs_target_raw_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 1 — Statistical Pre-Screening

Before fitting any model, the feature space needs to be cleaned up in two steps. The goal is to hand the model a set of features that are well-distributed and not redundant with each other.

### 1a · Skewness Test & Log₁₊ Transformation

Many of the World Bank indicators (population counts, GDP figures, infrastructure metrics) are heavily skewed with long tails. Linear models assume approximately normally distributed predictors, so leaving these uncorrected can distort coefficient estimates and inflate residuals.

For any feature where |skew| > 0.5, a log₁₊ transform is *tested*: `log1p(x + offset)`, where the offset is `max(0, -min(x))` to handle features that include zero or negative values. The transform is only applied if it actually reduces skewness — if the transformed column ends up more skewed than the original, the column is left unchanged. This prevents the correction from backfiring on features where a log scale is inappropriate (e.g. already-normalised survey scores or bounded rates).

In [ ]:
def screen_and_transform_skew(data, features, skew_thresh=SKEW_THRESH):
    # For each feature with |skew| > threshold, test the log1p transform.
    # Only apply if it actually reduces |skew| — if it makes things worse, keep the original.
    
    data_t = data.copy()
    transformed = []
    plot_info = []

    for feat in features:
        skew_val = data_t[feat].skew()
        if abs(skew_val) <= skew_thresh:
            continue

        offset = max(0.0, -data_t[feat].min())
        candidate = np.log1p(data_t[feat] + offset)

        if abs(candidate.skew()) < abs(skew_val):
            data_t[feat] = candidate
            transformed.append(feat)
            plot_info.append((feat, offset, skew_val))

    if not plot_info:
        print('No features transformed.')
        return data_t, transformed

    n = len(plot_info)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 6), squeeze=False)

    for i, (feat, offset, orig_skew) in enumerate(plot_info):
        original = np.expm1(data_t[feat]) - offset

        axes[0, i].hist(original, bins=40, color='#e07b54', edgecolor='white')
        axes[0, i].set_title(f'{feat} | before |skew|={abs(orig_skew):.2f}', fontsize=8)

        axes[1, i].hist(data_t[feat], bins=40, color='#5b8db8', edgecolor='white')
        axes[1, i].set_title(f'{feat} | after |skew|={abs(data_t[feat].skew()):.2f}', fontsize=8)

    fig.suptitle('Skewness Correction', fontweight='bold')
    plt.tight_layout()
    os.makedirs('plots', exist_ok=True)
    plt.savefig('plots/HZ_skewness_correction_histograms.png', dpi=150, bbox_inches='tight')
    plt.show()

    return data_t, transformed


df_t, transformed_cols = screen_and_transform_skew(df, ALL_FEATURES)

# skewness before/after table
pd.DataFrame({
    'Skew (original)': df[ALL_FEATURES].skew(),
    'Skew (after)'   : df_t[ALL_FEATURES].skew(),
    'Transformed'    : [f in transformed_cols for f in ALL_FEATURES]
}).sort_values('Skew (original)', key=abs, ascending=False)

In [ ]:
# scatter after log1p transform
ncols = 4
nrows = (len(ALL_FEATURES) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3))
axes = axes.flatten()

for i, feat in enumerate(ALL_FEATURES):
    axes[i].scatter(df_t[feat], df_t[TARGET], alpha=0.4, s=14, color='#e07b54', edgecolors='none')
    axes[i].set_xlabel(feat, fontsize=8)
    axes[i].set_ylabel(TARGET, fontsize=8)
    axes[i].set_title(feat, fontsize=9, fontweight='bold')
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f'Feature vs {TARGET} \u2014 After log\u2081\u208a Transform', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/HZ_feature_vs_target_post_transform_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### 1b · Variance Inflation Factor (VIF) Pruning

Many World Bank indicators are highly correlated by construction — GDP per capita, internet access, and life expectancy all tend to rise together. Including all of them would result in a classic case of colinearity, which would make the models unrelaiable.

VIF measures how much of a feature's variance is explained by the other features. A VIF above 10 means the feature is near-linearly predictable from its neighbours — it's contributing noise, not a unique signal. The pruning loop iteratively drops the worst offender until every remaining feature is below the threshold.

This process will be done for each region, and this is because the variations between variables might be different in different regions.

In [ ]:
def vif_prune(data, features, vif_thresh=VIF_THRESH, verbose=True):
    remaining = features.copy()

    while True:
        X_mat  = data[remaining].values
        vifs   = [variance_inflation_factor(X_mat, i) for i in range(len(remaining))]
        vif_df = pd.DataFrame({'Feature': remaining, 'VIF': vifs}).sort_values('VIF', ascending=False)

        max_vif  = vif_df['VIF'].iloc[0]
        max_feat = vif_df['Feature'].iloc[0]

        if max_vif <= vif_thresh:
            break

        if verbose:
            print(f"  drop '{max_feat}'  (VIF = {max_vif:.2f})")
        remaining.remove(max_feat)

    if verbose:
        print(f"All VIF <= {vif_thresh}, {len(remaining)} features remain.")

    return remaining, vif_df.reset_index(drop=True)


final_features, vif_table = vif_prune(df_t, ALL_FEATURES)
vif_table

---
## Phase 2 — `find_champion_model(X, y)`

Now, to create the models. Rather than setting a model type upfront, a grid search is run over two axes simultaneously: the polynomial degree (1, 2, or 3), as well as the  regularisation type (none vs. Ridge with varying strength).

Polynomial expansion allows the model to capture non-linear relationships and interaction effects between features, which will be helpful since alot of the correlations are indeed non-linear. 
Ridge regularisation penalises large coefficients, which is important when polynomial expansion increases the feature count significantly, reuslting in more noise and a higher chance of overfitting.

### Clarifications
* All configurations are evaluated on 5-fold cross-validated MSE. ```gridSearchCV()``` trains all the model variations on 80% of the data, and tests it on the other 20%, does this process with 5 different 20% slices, and takes the average MSE score
* ```PolynomialFeatures()``` has interactions at degrees higher than one; it it multiplies variables eith each other, thus interactions.
* The pipeline wraps all three steps (normalize, generate polynomial terms, fit model) into one object so gridSearchCV() can redo everything fresh on each training fold. Without it, you'd have to write the CV loop manually to avoid the normalization step "seeing" hidden test data. Also it makes the code much cleaner.
* ```include_bias=False``` tells ```PolynomialFeatures()``` not to add an intercept column with ones, and we do this because ```LinearRegression``` and ```Ridge``` already add their own intercept internally — so you'd have a duplicate.

In [ ]:
def find_champion_model(X, y, cv=5, verbose=True, max_degree=3):
    # grid: degree in {1..max_degree} x {LinearRegression, Ridge(alpha in logspace(-3,3,7))}
    degrees = range(1, max_degree + 1)
    alphas  = np.logspace(-3, 3, 7)     # 0.001, 0.01, 0.1, 1, 10, 100, 1000
    
    base = Pipeline([
        ('scaler', StandardScaler()),
        ('poly',   PolynomialFeatures(include_bias=False)),
        ('model',  LinearRegression())
    ])

    param_grid = []
    for degree in degrees:
        param_grid.append({'poly__degree': [degree], 'model': [LinearRegression()]})
    for degree in degrees:
        for alpha in alphas:
            param_grid.append({'poly__degree': [degree], 'model': [Ridge(alpha=alpha)]})

    gs = GridSearchCV(base, param_grid, scoring='neg_mean_squared_error',
                      cv=cv, refit=True, n_jobs=-1)
    gs.fit(X, y)

    best_model  = gs.best_params_['model']
    best_degree = gs.best_params_['poly__degree']
    best_mse    = -gs.best_score_
    model_name  = type(best_model).__name__

    if model_name == 'Ridge':
        desc = f"Ridge, Degree {best_degree}, Alpha {best_model.alpha:.3f}"
    else:
        desc = f"LinearRegression, Degree {best_degree}"

    if verbose:
        print(f"  Champion : {desc}")
        print(f"  CV MSE   : {best_mse:.4f}  (RMSE ≈ {best_mse**0.5:.4f})")

    results_df = (
        pd.DataFrame(gs.cv_results_)
        [['param_poly__degree', 'param_model', 'mean_test_score', 'std_test_score']]
        .assign(CV_MSE=lambda d: -d['mean_test_score'])
        .sort_values('CV_MSE')
        .reset_index(drop=True)
    )
    return gs.best_estimator_, desc, results_df


---
## Phase 3 — `calculate_importance(pipeline, X_test, y_test)`

Once a champion model is fitted, the question becomes: which features are actually doing the work?

Coefficient magnitudes from a polynomial Ridge model aren't directly interpretable — they depend on scale, interact across expanded terms, and change with regularisation strength. 

Instead, permutation importance is used: each feature is randomly shuffled (row-wise) (breaking its relationship with the target), the model is re-trains with this shuffled column, and the resulting drop in model performance is measured. A large drop means the model was genuinely relying on that feature; a small drop means the model barely noticed and thus is not important. This process is done hundreds of times

In [ ]:
def calculate_importance(pipeline, X_test, y_test, n_repeats=5000, title='Permutation Feature Importance', top_n=None, show_plot=True):
    result = permutation_importance(
        pipeline, X_test, y_test,
        n_repeats=n_repeats,
        scoring='neg_mean_squared_error',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    importance_df = (
        pd.DataFrame({
            'Feature'         : X_test.columns.tolist(),
            'Importance Mean' : result.importances_mean,
            'Importance Std'  : result.importances_std,
        })
        .sort_values('Importance Mean', ascending=False)
        .reset_index(drop=True)
    )

    fig = None
    if show_plot:
        plot_df = (importance_df.head(top_n) if top_n else importance_df).iloc[::-1]
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(plot_df['Feature'], plot_df['Importance Mean'], xerr=plot_df['Importance Std'], color='#c0392b')
        ax.axvline(0, color='black', linestyle='--')
        ax.set_xlabel('Mean Increase in MSE (when feature is shuffled)')
        ax.set_title(title, fontweight='bold')
        plt.tight_layout()
        os.makedirs('plots', exist_ok=True)
        safe_title = title.lower().replace(' ', '_').replace('—', '').replace('/', '').strip('_')
        plt.savefig(f'plots/HZ_{safe_title}.png', dpi=150, bbox_inches='tight')
        plt.show()

    return importance_df, fig

---
## Phase 4 — Execution & Comparison

`run_pipeline()` ties Phases 1–3 together for a given data slice. It handles the train/test split, calls the champion search, evaluates on the hold-out set, and runs permutation importance — returning everything needed for the summary and heatmap.

### 4a · Global Run

The first run uses the full dataset. The feature set entering here (`final_features`) is the output of Phase 1 applied globally — skewness-corrected and VIF-pruned. This gives a baseline picture of which indicators best predict happiness across all countries and years combined.

In [ ]:
def run_pipeline(data, features, target=TARGET, label='Global', n_repeats_perm=500, show_perm_plot=True):
    X = data[features]
    y = data[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

    # Cap polynomial degree to prevent term explosion on small training sets
    n_train = len(X_train)
    if n_train < 60:
        max_degree = 1
    elif n_train < 120:
        max_degree = 2
    else:
        max_degree = 3

    print(f"\n{label}  —  {len(data)} rows | {len(features)} features | max_degree={max_degree}")

    champion, desc, _ = find_champion_model(X_train, y_train, max_degree=max_degree)

    preds    = champion.predict(X_test)
    test_mse = mean_squared_error(y_test, preds)
    test_r2  = r2_score(y_test, preds)

    # Cross-validated R² on full dataset — more stable than a single split for small n
    cv_folds  = max(3, min(5, len(X) // 10))
    cv_r2_arr = cross_val_score(clone(champion), X, y, cv=cv_folds, scoring='r2')
    cv_r2     = cv_r2_arr.mean()
    print(f"  CV R²    : {cv_r2:.4f}  (±{cv_r2_arr.std():.4f}, {cv_folds}-fold)")

    imp_df, _ = calculate_importance(
        champion, X_test, y_test,
        n_repeats=n_repeats_perm,
        title=f'Permutation Importance — {label}',
        show_plot=show_perm_plot,
    )

    return {'label': label, 'champion_desc': desc, 'importance_df': imp_df,
            'test_mse': test_mse, 'test_r2': test_r2, 'cv_r2': cv_r2, 'features_used': features}


global_result = run_pipeline(df_t, final_features, label='Global')

### 4b · Regional Loop

The global model averages over enormous variation — a factor that predicts happiness in Western Europe may be irrelevant or even inversely related in Sub-Saharan Africa. This makes our insights too general for any specific part of the world, so the whole process of VIF pruning & champion search is redone for each region.

Regions with fewer than 30 samples are skipped — not enough data to support training & 5-fold CV split reliably.

In [ ]:
MIN_REGION_SAMPLES = 30
regional_results   = {}

for region in sorted(df_t['Regional Indicator'].unique()):
    subset = df_t[df_t['Regional Indicator'] == region].copy()

    if len(subset) < MIN_REGION_SAMPLES:
        print(f"skip '{region}' — {len(subset)} samples")
        continue

    reg_final, _ = vif_prune(subset, ALL_FEATURES, verbose=False)
    if len(reg_final) < 1:
        print(f"skip '{region}' — nothing survived VIF")
        continue

    regional_results[region] = run_pipeline(subset, reg_final, label=region, show_perm_plot=False)

print(f"Done — {len(regional_results)} regions processed.")

### 4b.5 · Spearman Correlation — Global & Regional

Before moving to the final summary, it's worth comparing permutation importance rankings against a simpler baseline: Spearman rank correlation. 

Spearman measures whether two variables move together monotonically — it's non-parametric and doesn't assume linearity, which makes it a a good fit for this dataset.

The comparison is useful because it reveals whether this pipeline we developed adds value or not. Because if permutation importance and Spearman rank the same features at the top, then this pipeline is useless. But if they diverge, It shows that our process that takes into account coliniarity and variable interaction was important to finding stronger insight.

The cell below prints Spearman rankings per scope. The one after it directly compares top-5 Spearman vs top-5 permutation importance to show where both methods agree and where the pipeline finds something Spearman misses (or vice versa).

In [ ]:
SIG_SPEARMAN_THRESH = 0.65

def print_top_spearman(data, features, scope_label):
    sp      = data[features + [TARGET]].corr(method='spearman')[TARGET].drop(TARGET)
    sp_abs  = sp.abs()
    sp_norm = sp_abs / sp_abs.max() if sp_abs.max() > 0 else sp_abs

    sig_idx = sp_norm[sp_norm >= SIG_SPEARMAN_THRESH].index
    if len(sig_idx) == 0:
        sig_idx = [sp_abs.idxmax()]

    ordered   = sp_abs.loc[sig_idx].sort_values(ascending=False).index
    feat_strs = [f"{f} ({sp[f]:.3f})" for f in ordered]
    print(f"  {scope_label:<40} {' > '.join(feat_strs)}")


print(f"Top Spearman \u03c1 features per scope (normalised |\u03c1| >= {SIG_SPEARMAN_THRESH:.2f}):")
print('\u2500' * 80)
print_top_spearman(df_t, ALL_FEATURES, 'Global')
for region in sorted(regional_results):
    subset = df_t[df_t['Regional Indicator'] == region]
    print_top_spearman(subset, ALL_FEATURES, region)

In [ ]:
TOP_N = 5

print("Scoring guide")
print("─" * 65)
print(f"  Match rate : how many of the top-{TOP_N} features appear in both lists.")
print(f"               1.0 = all {TOP_N} agree  |  0.0 = no overlap.")
print(f"  ✓ marks features that appear in both lists.")
print()

scope_data = [('Global', df_t, global_result)] + [
    (region, df_t[df_t['Regional Indicator'] == region], regional_results[region])
    for region in sorted(regional_results)
]

for scope_label, subset, result in scope_data:
    sp     = subset[ALL_FEATURES + [TARGET]].corr(method='spearman')[TARGET].drop(TARGET)
    sp_top = sp.abs().sort_values(ascending=False).head(TOP_N).index.tolist()
    pm_top = result['importance_df'].head(TOP_N)['Feature'].tolist()

    common     = set(sp_top) & set(pm_top)
    match_rate = len(common) / TOP_N

    sp_str = '  |  '.join(f"{'✓ ' if f in common else ''}{f}" for f in sp_top)
    pm_str = '  |  '.join(f"{'✓ ' if f in common else ''}{f}" for f in pm_top)

    print(f"── {scope_label} ──")
    print(f"  Spearman : {sp_str}")
    print(f"  Pipeline : {pm_str}")
    print(f"  Match    : {len(common)}/{TOP_N} ({match_rate:.0%})")
    print()

Safe to say the results were very different.

### 4c · Champion Model Summary

In [ ]:
all_results = [global_result] + [regional_results[r] for r in sorted(regional_results)]

summary_df = pd.DataFrame([
    {
        'Scope'           : res['label'],
        'Champion Model'  : res['champion_desc'],
        'Test RMSE'       : round(res['test_mse'] ** 0.5, 4),
        'Test R²'         : round(res['test_r2'], 4),
        'CV R²'           : round(res['cv_r2'], 4),
        'Features Used'   : len(res['features_used']),
        'Top-3 Predictors': ' > '.join(res['importance_df'].head(3)['Feature'].tolist()),
    }
    for res in all_results
])

pd.set_option('display.max_colwidth', 80)
summary_df


> **Note on CV R² vs Feature Importance:** Several regions show negative cross-validated R² scores (e.g. South Asia: −2.48, Middle East: −0.53, East Asia: −0.34). A negative CV R² means the model generalises worse than simply predicting the regional mean on held-out folds — a sign of overfitting driven by small sample sizes (49–133 rows in affected regions) relative to the polynomial feature space.
>
> This does **not** contradict the feature importance findings. CV R² measures predictive generalisation on unseen data; permutation importance measures which features the *fitted* model relies on. Even a model that overfits on small data can correctly identify the direction and relative ranking of drivers — and the heatmap results are consistent with established literature (e.g. governance quality in Western countries, health and social indicators in South/Southeast Asia). For data-limited regions, treat the feature rankings as informative signals about *what matters* rather than as precise predictive coefficients.

### 4d · Feature Importance Heatmap — Global vs Regional

The heatmap brings everything together. Each row is a scope (global or a region), each column is a feature that appeared in at least one run's final feature set. Cell values are raw permutation importance means; colour is row-normalised so that each scope's strongest predictor reads as 1.0, making cross-scope patterns visible regardless of differences in absolute importance scale.

Features are ordered left-to-right by global importance, so the most universally impactful predictors appear first. Features that appear warm across many rows are consistent global drivers; features that are warm in only one or two rows are region-specific signals that the global model would have missed or underweighted.

In [ ]:
all_feat_names = sorted({feat for res in all_results for feat in res['importance_df']['Feature']})

heatmap_df = pd.DataFrame(
    {
        res['label']: {
            f: dict(zip(res['importance_df']['Feature'], res['importance_df']['Importance Mean'])).get(f, 0.0)
            for f in all_feat_names
        }
        for res in all_results
    },
    index=all_feat_names
).T

heatmap_norm = heatmap_df.div(heatmap_df.max(axis=1).replace(0, 1), axis=0)

# order columns by global importance
global_order = heatmap_df.loc['Global'].sort_values(ascending=False).index.tolist()
heatmap_df   = heatmap_df[global_order]
heatmap_norm = heatmap_norm[global_order]

n_rows, n_cols = len(heatmap_df), len(heatmap_df.columns)
fig, ax = plt.subplots(figsize=(max(14, n_cols * 1.1), max(5, n_rows * 0.7)))

sns.heatmap(
    heatmap_norm,
    annot=heatmap_df.round(3), fmt='.3f',
    cmap='Blues', vmin=0, vmax=1,
    linewidths=0.5, linecolor='#dddddd',
    ax=ax,
    cbar_kws={'label': 'Normalised Importance (row-max = 1)'},
    annot_kws={'size': 7},
)

ax.set_title(
    'Feature Importance Heatmap \u2014 Global vs Regional\n'
    '(values = raw permutation importance; colour = row-normalised)',
    fontsize=13, fontweight='bold', pad=16
)
ax.set_xlabel('Feature', fontsize=10)
ax.set_ylabel('Scope', fontsize=10)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0,  labelsize=9)

plt.tight_layout()
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/HZ_feature_importance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nStrongest global predictors:")
for i, feat in enumerate(global_order[:5], 1):
    print(f"  {i}. {feat}  ({heatmap_df.loc['Global', feat]:.4f})")


# top significant features per scope
SIG_NORM_THRESH = 0.4
print(f'\nTop features per scope (normalised importance >= {SIG_NORM_THRESH}):')
print('\u2500' * 65)
for scope in heatmap_norm.index:
    row = heatmap_norm.loc[scope]
    sig = row[row >= SIG_NORM_THRESH].sort_values(ascending=False)
    if sig.empty:
        sig = row.sort_values(ascending=False).head(1)
    feat_strs = [f"{f} ({heatmap_df.loc[scope, f]:.3f})" for f in sig.index]
    print(f"  {scope:<40} {' > '.join(feat_strs)}")